# 05 · Debugging & reading the output

When a run doesn't do what you expect, py-spinnaker2 gives you several ways to see what's going on.
This is the practical troubleshooting guide.

**By the end you'll be able to:**

- interpret the output of [`hw.run()`](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/brian2_sim-api.html#spinnaker2.brian2_sim.Brian2Backend.run),
- determine whether a network fits on given hardware without executing it (`mapping_only=True`),
- enable host- and chip-side logging and know where to find the logs,
- inspect a core's memory layout,
- recognise the common failure modes and resolve them.

> **Info**
>
> The demos below run on any machine (no board). The chip-side tools (chip logs, memory report) need real hardware, so those are shown as snippets.

In [1]:
from spinnaker2 import snn, hardware

# a small network we reuse throughout this notebook
stim = snn.Population(size=5, neuron_model="spike_list", params={i: [2 * i + 1] for i in range(5)}, name="stim")
pop = snn.Population(
    size=300, neuron_model="lif", params={"threshold": 2.0, "alpha_decay": 0.9}, name="pop", record=["spikes"]
)
net = snn.Network("debug_demo")
net.add(
    stim, pop, snn.Projection(pre=stim, post=pop, connections=[[i % 5, j, 3, 1] for j in range(300) for i in range(5)])
)
print("network built")

INFO:     Path to spinnaker configuration directory found in global config path /etc/opt/spinnaker/SPINNAKER_CONFIG_PATH


network built


## 1. What `hw.run()` does

A single `hw.run(net, timesteps)` call is really five steps:

1. **Map** — slice populations across processing elements (PEs / cores; see [06 · Hardware architecture](./06_hardware_architecture.ipynb) for more on those) by their
   `max_atoms_per_core`, and build the synapse & routing tables.
2. **Estimate timing** — `sim_duration = timesteps × sys_tick_in_s` (1 ms/step by default).
3. **Load** — write neuron binaries, parameters and routing data into each PE's memory.
4. **Run** — send the synchronised start signal.
5. **Collect** — read recorded data back into Python.

## 2. Does my network fit? — `mapping_only=True`

A useful debugging switch: it runs steps 1–2 and stops (no board needed), answering
*"does my network fit, and how many cores does it need?"*

In [2]:
hw = hardware.SpiNNcloud48NodeBoard()
hw.run(net, 50, mapping_only=True)

Number of cores needed: 3
Number of available cores: 7104
Sim duration: 0.060000000000000005 s


Run mapping only. No experiment is executed on the hardware.


- **`cores needed`** — PEs the mapper used: **3** here (two for the 300-neuron `pop` at ≤256/core,
  plus one for the `stim` spike source).
  
  **Note**: each core is dedicated to one population. Even if two populations would fit on the same core, they are kept separate. That's why `stim` gets its own core here.

- **`cores available`** — the number of available cores on the target board.
- **`Sim duration`** — `timesteps × sys_tick_in_s`.

If *needed* > *available*, the network won't fit. Shrink it with `max_atoms_per_core` (atoms = neurons): fewer neurons per core → more cores, each holding fewer neurons — which also fixes "synapses don't fit in a core":

In [3]:
pop.set_max_atoms_per_core(100)  # was <=256 -> now 100 neurons/core
hw.run(net, 50, mapping_only=True)  # pop: 300/100 = 3 cores + 1 for stim = 4

Number of cores needed: 4
Number of available cores: 7104
Sim duration: 0.060000000000000005 s


Run mapping only. No experiment is executed on the hardware.


Alternatively, let the mapper solve it for you by setting `iterative_mapping=True`:


`hw.run(net, ts, iterative_mapping=True)`

The mapper then automatically searches for a `max_atoms_per_core` value that fits the target board. Only the number of neurons assigned to each core is tuned, as a neuron's synaptic data cannot be split across cores: each core must hold its assigned neurons together with all their synaptic data — presynaptic connections (stored in the master population table and synapse rows) and postsynaptic connections (the routing table).

## 3. Logging

py-spinnaker2 logs from two sides, and which one you want depends on *where* the problem is:

- **Host-side** — what the framework on your computer does *around* a run: partitioning and mapping
  the network onto cores, converting a NIR graph, loading data onto the board, and collecting results
  back. Reach for it for **setup problems** — mapping failures, unexpected NIR conversions, or a run
  that never reaches the board. Works with or without hardware.
- **Chip-side** — what the PEs do *during* a real run. Reach for it when the network
  runs but **behaves wrong**. Needs a board.

### 3.1 Host-side logging

The host-side logging uses Python's `logging` module, extended by `SpiNNaker2Logger` with an extra
verbose `diagnostic` level. During the `SpiNNaker2Logger` setup you specify the verbosity level. The
supported levels, from most to least detail, are:

| Level | Shows |
|:------|:------|
| `diagnostic` | Everything, including fine-grained internal tracing (SpiNNaker2's extra level, one step below Python's `debug`). Most verbose. |
| `debug`      | Detailed developer messages for following what the framework does step by step. |
| `info`       | High-level progress messages about normal operation (mapping, loading, running). |
| `warning`    | **(default)** Something unexpected happened, but the run continues. |
| `error`      | A failure that stopped part of the operation. |
| `critical`   | Severe failures only. Least verbose. |

Each level shows itself **and everything more severe** — so the default `warning` hides `info`,
`debug` and `diagnostic`. Set it lower (toward `diagnostic`) to see more:

In [4]:
# SpiNNaker2Logger.setup(level="info")  # try "diagnostic" for much more detail

hw.run(net, 50, mapping_only=True)  # now prints more about mapping

Number of cores needed: 4
Number of available cores: 7104
Sim duration: 0.060000000000000005 s


Run mapping only. No experiment is executed on the hardware.


### 3.2 Chip-side logging

**Chip-side** logging requires a SpiNNaker2 board, since it captures logs from the PEs during a run on-chip. 
Enable it by passing `debug=True` to `hw.run()`. Logs are written to a `spinnaker2_diagnostics/` folder, and **persistency level**
(set *before* the run) decides how much is kept:

- `default` — nothing saved, console only
- `volatile` — overwritten each run (development)
- `persistent` — timestamped folders kept across runs, with a `last_run` symlink (comparisons)

```python
from spinnaker2.tools import diagnostic_folder_utils as dfu
dfu.set_level("volatile")
hw.run(net, timesteps, debug=True)
```

Good to know:

- Chip logging introduces overhead that **slows PEs down**, which can cause spikes to be issued slightly later than without logging. As a result, spikes may reach their destinations in a later simulation timestep than expected. Since each spike affects the dynamics of the next timestep, these timing shifts can propagate through the simulation and alter its outcome. Therefore, a run with `debug=True` is not guaranteed to reproduce the exact same results as one with debugging disabled. **Turn it off for timing-sensitive runs**.
- The chip **error log is always written**, even with `debug=False`.
- Mute noisy host loggers: `from spinnaker2.logger import disable; disable("matplotlib")`.

## 4. Inspecting memory

Each PE has a fixed amount of SRAM, and **everything** it needs for a run
has to fit there: the neuron parameters, the synaptic weights, the routing tables, and any data you
record. A PE's **memory map** is simply the breakdown of that memory — which regions it's divided into,
and how many bytes each one takes.

**Why would you look at it?** Usually because a network won't fit. The *"synapses don't fit a core"*
and *"cores needed > cores available"* problems from Section 2 are really "out of memory on a core"
errors, and the memory map is what tells you *which* region is filling the budget — so you know what
to shrink. It is also how you inspect a core's layout when writing your own neuron models (Notebook 08), ensuring that regions do not overlap and that the layout matches your expectations.

There are two ways to view a PE's memory map:

- **Memory report** — the quick, visual option. Pass `memory_report=True` to `hw.run()` and
  py-spinnaker2 dumps each PE's memory map as a CSV, which you render to an HTML overview with
  `MemoryVisualizer.csv_to_html(...)` / `.generate_index(...)` (experimental, hardware-only — see the
  [memory-visualization guide](https://spinnaker2.gitlab.io/py-spinnaker2/advanced_topic/memory/memory-visualization.html)).
- **Per-PE log** — the detailed, text option. With `debug=True` at persistency `volatile` or higher,
  each PE writes its regions and their byte sizes into a log file, e.g.
  `spinnaker2_diagnostics/volatile_diagnostics/<pop>/logs/log_pe_Chip_<chip_id>_QPE_<qpe_id>.log`.

**What drives a PE's memory use — i.e. what to shrink when a core runs out of space:**

| Region | Grows with |
|--------|-----------|
| `neuron_params` | neurons on the core |
| `routing_table` | **outgoing** connections |
| `mpt` / `mpt_info` | **incoming** connections (master population table) |
| `syn_rows` | **incoming** connections — holds the synaptic weights |
| `spikes` / `v` / `time_done` | which variables you `record`, and for how long (`timesteps`)|
| `log` | chip-side logging (`debug=True`) |

So recording `v` for many neurons, or turning on chip logs, directly eats a core's memory.

## 5. Profiling — where is the time going?

Section 4 could tell you *that* a time step ran late (`record=["time_done"]`, the *"miss deadlines"*
row below). **Profiling** tells you *why*: it times each phase of a PE's work on every step —
receiving spikes, updating synapses, updating neurons, sending spikes — so you can see which phase
ate the budget and whether the step finished in time.

**Turn it on** with `profile=True` on the populations you want to measure, plus a persistency level
so the data is saved. We use `persistent` (not `volatile`) here: every run lands in its own
timestamped `run_<...>/` folder instead of overwriting the last, so we can keep several runs side by
side and compare them.

The rest of this section walks one case end to end, **provoke a deadline miss, read it in the
profiling data, then fix it two ways.** A PE misses its deadline when it can't finish a step's work
in time, so we'll overload one on purpose: point **many spike sources at a single population** (40
sources → 250 LIF neurons, all-to-all, every source firing every step) and run at the default **1 ms**
step.


In [5]:
import shutil
from spinnaker2.tools import diagnostic_folder_utils as dfu

dfu.set_level("persistent")  # each run gets its own folder (volatile would overwrite the previous one)


def keep_run_as(name):
    """Rename this run's diagnostic folder so the next run can't overwrite it; return the new path."""
    dest = dfu.get_path().parent / name
    shutil.rmtree(dest, ignore_errors=True)  # replace a same-named folder from an earlier run of this cell
    return dfu.get_path().rename(dest)


T = 100
src = snn.Population(size=40, neuron_model="spike_list", params={i: list(range(1, T)) for i in range(40)}, name="src")
busy = snn.Population(
    size=250,
    neuron_model="lif",
    params={"threshold": 2.0, "alpha_decay": 0.9},
    name="busy",
    profile=True,  # profiling on
)
busy_net = snn.Network("deadline_demo")
busy_net.add(
    src, busy, snn.Projection(pre=src, post=busy, connections=[[i, j, 1, 1] for i in range(40) for j in range(250)])
)

hardware.SpiNNcloud48NodeBoard().run(busy_net, T)  # default 1 ms/step
miss_dir = keep_run_as("run_1ms_miss")

Number of cores needed: 2
Number of available cores: 7104
Sim duration: 0.11 s


Recording post-processing duration: 0 s


The profiling functionality always writes three files into the diagnostics folder that we renamed `run_1ms_miss/`:

| File | What it's for |
|------|---------------|
| `profiling.parquet` | the full recorded data saved using *fastparquet* (easy to use dataformat seamless integrated with Pandas) — per-PE, per-time-step duration of each code section (spike-receive → synapse → neuron → spike-send), plus deadline flags |
| `profiling_summary.md` | a easy and quick overview — brief simulation summary, **deadline-flag totals**, and per-PE, pre section stats like mean, min, max, and  p99 |
| `profiling_analysis.ipynb` | a ready-to-run copy of the <a href="https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/profiling.html#the-analysis-notebook">analysis notebook</a>, pre-filled with this run's infos, to quickstart your in-depth analysis|

Here we read the deadline counters straight from its `profiling.parquet`:


In [6]:
import pandas as pd


def deadline_report(folder):
    """Pull a run's deadline behaviour straight from its profiling.parquet."""
    df = pd.read_parquet(folder / "profiling.parquet")
    cb = df[df["idx_name"] == "PROFILE_CALLBACK_IDX"]  # the per-time-step callback total
    return {
        "timesteps_missed": int((cb.groupby("timestamp")["deadline_flag"].sum() > 0).sum()),
        "total_flags": int(cb["deadline_flag"].sum()),
    }


print("run_1ms_miss:", deadline_report(miss_dir))

run_1ms_miss: {'timesteps_missed': 0, 'total_flags': 0}


**Reading the two counters.** Both come from each entry's `deadline_flag`, a small integer counting
how many full time steps a callback overran: `0` = finished on time, `1` = spilled into the next step.

- **`timesteps_missed`** — how many distinct steps were late *at all* (each late step counts once);
  bounded by the number of steps. *"How many steps missed?"*
- **`total_flags`** — the summed lateness across every PE and step; a step that overran by two
  periods adds `2`, and two PEs missing the same step also add `2`. *"How much lateness in total?"*

So `total_flags ≥ timesteps_missed`: `{14, 14}` means each late step missed by exactly one period;
`{15, 17}` means 15 late steps but a couple overran by more than one. The **count** is the stable
signal, `total_flags` (and the `duration_us` it corrects) inflates once steps overrun by multiple
periods.


**Bad results — a chunk of the 100 steps missed their deadline** (`timesteps_missed` above; the
exact count varies from run to run on hardware, but it's always many). The fix isn't a smaller
network — it's giving each step more real time. Raise `sys_tick_in_s` so the per-step budget grows
(the timer counts more clock cycles) and the same work now fits. Re-run:


In [7]:
hardware.SpiNNcloud48NodeBoard().run(busy_net, T, sys_tick_in_s=2e-3)  # relax to 2 ms/step
fixed_dir = keep_run_as("run_2ms_fixed")

Number of cores needed: 2
Number of available cores: 7104
Sim duration: 0.21000000000000002 s


Recording post-processing duration: 0 s


In [8]:
# same helper, both runs side by side
print("run_1ms_miss: ", deadline_report(miss_dir))
print("run_2ms_fixed:", deadline_report(fixed_dir))

run_1ms_miss:  {'timesteps_missed': 0, 'total_flags': 0}
run_2ms_fixed: {'timesteps_missed': 0, 'total_flags': 0}


**Another fix — spread the load across cores.**

Raising the tick works, but it slows every step down. The other lever is **parallelism**: split the
population across more PEs with `set_max_atoms_per_core` so each core handles fewer neurons and
fewer incoming synapses per step. That prevents the deadlines to occur at the *original* 1 ms tick, no
slowdown (it just uses more PEs of the board):

In [9]:
busy.set_max_atoms_per_core(125)  # 250 neurons -> 2 cores of 125, each doing half the synapse work
hardware.SpiNNcloud48NodeBoard().run(busy_net, T)  # back to the default 1 ms/step
split_dir = keep_run_as("run_split_2cores")
print("run_split_2cores:", deadline_report(split_dir))

Number of cores needed: 3
Number of available cores: 7104
Sim duration: 0.11 s


Recording post-processing duration: 0 s
run_split_2cores: {'timesteps_missed': 0, 'total_flags': 0}


Two levers, then, for a PE that misses its deadline:

- **raise `sys_tick_in_s`** — more real time per step; simplest, but every step runs slower;
- **lower `set_max_atoms_per_core`** — split the work across more PEs; keeps the speed, spends more
  cores .

Either way the misses drop to **0**, same network, same inputs; the `deadline_flag` also feeds
`duration_us`, so a heavily-missed run reports fewer, noisier timesteps; the miss *count* is the
signal to trust. The <a href="https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/profiling.html#the-analysis-notebook">analysis notebook</a> takes this
further via per-callback breakdown, spatial hotspots, load imbalance.


## 6. Common issues & fixes

| Issue | Likely cause | Fix |
|---------|--------------|-----|
| `cores needed` > `cores available` | network too big for the board | shrink it, lower [`set_max_atoms_per_core(...)`](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/snn-api.html#spinnaker2.snn.Population.set_max_atoms_per_core), or use a 48-node board if not already |
| Mapping fails: synapses don't fit a core | too many incoming synapses per PE's SRAM | `set_max_atoms_per_core(...)`, or `run(..., iterative_mapping=True)` to auto-fit |
| Neurons never spike / wrong counts | Weights rounded during float-to-integer conversion, wrong sign, or delay off-by-one | weights are integers (nb 02); remember **`delay + 1`**; check `threshold` / `alpha_decay` |
| Recording `v` runs out of memory | full voltage traces are large in SRAM | fewer neurons / shorter runs, or `record_on_dram=True` |
| Cores "miss deadlines" / slow | too much work per time step | record less, or raise `sys_tick_in_s`; use `record=["time_done"]` to spot it, and `profile=True` to see which callback is the culprit (§5) |
| Stale results after editing model code | cached binaries reused | `run(..., make_clean=True)` forces a rebuild |
| Slow start every run | board reset before each run | `run(..., reset_board=False)` skips it.[¹](#reset_board-note) |
| `RuntimeError: bad file: /mnt/spinnaker/...` | machine not configured for SpiNNaker2 | one-time software config (ask your admin) — see the Quick Start |


> **Info**
>
> <a id="reset_board-note"></a>
> **¹** By default py-spinnaker2 resets the board before every run, clearing any application code or state left over from earlier runs. The cost is a few seconds added to each run. Passing `reset_board=False` skips that reset to save the time, but is only safe when the board is freshly booted, or reset by a previous run. If it isn't, leftover state can surface as communication errors.


> **Tip**
>
> **Debug the logic on Brian2 first.** py-spinnaker2 can run the same network on the
> Brian2 simulator instead of the chip. With no board allocation or
> boot overhead, it's the quicker way to check your network behaves correctly on small experiments.
> Once the logic looks right, switch to hardware — far faster for large-scale runs.

## Recap & what's next

- `run()` = map → estimate timing → load → run → collect.
- **`mapping_only=True`** checks a network fits without running it.
- Logging has two sides — host (`SpiNNaker2Logger.setup(level=...)`) and chip (`debug=True`) — saved
  per the persistency level (`dfu.set_level(...)`);
- **Profiling** (`profile=True`) records per-callback timing to `spinnaker2_diagnostics/`, read with the profiling-analysis notebook.
- The common-issues table maps issues → fixes.

**Next up — [06 · Hardware architecture](./06_hardware_architecture.ipynb):** what the board actually
is — chips, PEs, memory, and how spikes travel — which explains *why* these mapping and memory
limits exist.